# Notebook 06 — Análisis y Predicción Temporal

**Proyecto:** Sistema de Predicción y Clasificación de la Desnutrición en niños menores de cinco años  
**Fase CRISP-DM:** 4 (Modelado) — Componente temporal  
**Datasets:** `serie_temporal_mensual.csv` y `serie_temporal_departamento.csv`

---
## Contenido
1. Carga y preparación de la serie temporal
2. Visualización de la evolución mensual
3. Análisis de tendencia
4. Análisis de estacionalidad
5. Modelos de proyección
   - 5a. Promedio móvil (baseline)
   - 5b. Regresión lineal con tendencia
   - 5c. SARIMA (si statsmodels disponible)
6. Proyección 2026 — próximos 6 meses
7. Análisis comparativo territorial (Cesar vs La Guajira vs Magdalena)
8. Hallazgos y limitaciones

## 1. Carga y preparación de la serie temporal

In [ ]:
import sys
!{sys.executable} -m pip install statsmodels --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})

# Cargar serie temporal mensual
st = pd.read_csv('../data/processed/serie_temporal_mensual.csv')
st = st[st['anio_mes'] <= '2025-12'].copy()  # excluir el punto 2026-01 (error digitación)
st['fecha'] = pd.to_datetime(st['anio_mes'])
st = st.sort_values('fecha').reset_index(drop=True)
st['t'] = np.arange(len(st))  # índice numérico para modelos

# Serie estable (2024-2025): 2023 tiene muy pocos casos/mes → tasas inestables
st_estable = st[st['anio_mes'] >= '2024-01'].copy().reset_index(drop=True)
st_estable['t'] = np.arange(len(st_estable))

# Cargar serie departamental
sd = pd.read_csv('../data/processed/serie_temporal_departamento.csv')

print(f'Serie completa    : {len(st)} puntos ({st["anio_mes"].min()} → {st["anio_mes"].max()})')
print(f'Serie estable     : {len(st_estable)} puntos (2024-2025)')
print(f'Serie departamental: {len(sd)} filas')
print(f'\nColumnas serie mensual:')
print(list(st.columns))
print(f'\nEstadísticas tasa_desnut (serie estable):')
print(st_estable['tasa_desnut'].describe().round(2))

---
## 2. Visualización de la evolución mensual

**Nota sobre 2023:** Solo tiene entre 1 y 21 casos por mes (captura parcial del año).
Con tan pocos casos, una tasa del 100% en enero o 25% en marzo no es informativa —
puede ser que solo llegaron 5 niños ese mes y 4 estaban desnutridos.
El análisis de tendencia se hace principalmente con 2024-2025.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# ── Gráfico 1: tasa + casos ───────────────────────────────────────────────
ax1 = axes[0]
ax2 = ax1.twinx()

ax2.bar(range(len(st)), st['casos'], color='#185fa5', alpha=0.2, label='Casos/mes')
ax1.plot(range(len(st)), st['tasa_desnut'], 'o-', color='#c0392b',
         linewidth=2, markersize=5, label='Tasa desnutrición (%)')

# Línea de tendencia solo para 2024-2025
x_estable = st_estable['t'].values.reshape(-1, 1)
lr = LinearRegression().fit(x_estable, st_estable['tasa_desnut'])
tend_vals = lr.predict(x_estable)
idx_start = st[st['anio_mes'] == '2024-01'].index[0]
ax1.plot(range(idx_start, idx_start + len(tend_vals)), tend_vals,
         '--', color='#8e44ad', linewidth=1.5, label=f'Tendencia 2024-25 ({lr.coef_[0]:+.2f}%/mes)')

# Separadores de año
for yr_idx in [12, 24]:
    ax1.axvline(yr_idx - 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    ax1.text(yr_idx - 0.3, 105, str(2023 + yr_idx//12), fontsize=9, color='gray')

ax1.set_xticks(range(len(st)))
ax1.set_xticklabels(st['anio_mes'], rotation=45, ha='right', fontsize=7)
ax1.set_ylabel('Tasa desnutrición (%)', color='#c0392b')
ax2.set_ylabel('Casos por mes', color='#185fa5')
ax1.set_ylim(0, 115)
ax1.set_title('Evolución mensual de la tasa de desnutrición — 2023 a 2025', fontsize=13, pad=12)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', framealpha=0.5, fontsize=9)

# ── Gráfico 2: zscore promedio ────────────────────────────────────────────
ax3 = axes[1]
ax3.plot(range(len(st)), st['zscore_pt_media'], 's-', color='#8e44ad',
         linewidth=2, markersize=5)
ax3.fill_between(range(len(st)), st['zscore_pt_media'], alpha=0.1, color='#8e44ad')
ax3.axhline(-2, color='#e67e22', linestyle='--', linewidth=1.2,
            label='Umbral desnut. moderada (z = −2)')
ax3.axhline(-3, color='#c0392b', linestyle='--', linewidth=1.2,
            label='Umbral desnut. severa (z = −3)')
ax3.axhline(0, color='gray', linewidth=0.7, alpha=0.4)
for yr_idx in [12, 24]:
    ax3.axvline(yr_idx - 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax3.set_xticks(range(len(st)))
ax3.set_xticklabels(st['anio_mes'], rotation=45, ha='right', fontsize=7)
ax3.set_title('Evolución mensual del z-score promedio (zscore_pt) — 2023 a 2025', fontsize=13, pad=12)
ax3.set_ylabel('Z-score promedio peso/talla')
ax3.legend(fontsize=9, framealpha=0.5)

plt.tight_layout()
plt.show()

print(f'Tasa promedio 2024: {st[st["anio_mes"].str.startswith("2024")]["tasa_desnut"].mean():.1f}%')
print(f'Tasa promedio 2025: {st[st["anio_mes"].str.startswith("2025")]["tasa_desnut"].mean():.1f}%')
print(f'Z-score promedio 2024: {st[st["anio_mes"].str.startswith("2024")]["zscore_pt_media"].mean():.3f}')
print(f'Z-score promedio 2025: {st[st["anio_mes"].str.startswith("2025")]["zscore_pt_media"].mean():.3f}')

---
## 3. Análisis de tendencia

Se ajusta una regresión lineal sobre la serie 2024-2025 para cuantificar si la tasa
de desnutrición está subiendo, bajando o estable.

- **Pendiente positiva** → la situación empeora con el tiempo
- **Pendiente negativa** → mejora
- **p-value > 0.05** → la tendencia no es estadísticamente significativa (podría ser ruido)

In [ ]:
from scipy import stats as scipy_stats

x = st_estable['t'].values
y = st_estable['tasa_desnut'].values

slope, intercept, r_val, p_val, std_err = scipy_stats.linregress(x, y)

print('=== Regresión lineal sobre tasa_desnut (2024-2025) ===')
print(f'Pendiente     : {slope:+.4f} puntos porcentuales por mes')
print(f'Intercepto    : {intercept:.2f}%')
print(f'R²            : {r_val**2:.4f}')
print(f'p-value       : {p_val:.4f}')
print()
if p_val < 0.05:
    direccion = 'CRECIENTE (empeora)' if slope > 0 else 'DECRECIENTE (mejora)'
    print(f'Tendencia estadísticamente significativa → {direccion}')
    print(f'A este ritmo, en 12 meses la tasa cambiaría ~{slope*12:+.1f} puntos porcentuales')
else:
    print('Tendencia NO estadísticamente significativa (p > 0.05)')
    print('La tasa se mantiene relativamente estable — no hay evidencia de mejora ni deterioro claro')
    print(f'Aunque la pendiente es {slope:+.4f}, podría ser variabilidad aleatoria')

# Comparar también zscore
slope_z, _, r_z, p_z, _ = scipy_stats.linregress(x, st_estable['zscore_pt_media'].values)
print(f'\n=== Tendencia zscore_pt_media (2024-2025) ===')
print(f'Pendiente : {slope_z:+.5f} z-score por mes')
print(f'p-value   : {p_z:.4f}')
print(f'Tendencia zscore: {"significativa" if p_z < 0.05 else "no significativa"}')

---
## 4. Análisis de estacionalidad

Se analiza si hay meses del año donde la tasa de desnutrición es consistentemente
más alta o más baja. Con solo 2 años de datos estables (2024-2025), la estacionalidad
detectada es indicativa pero no concluyente — se necesitarían al menos 3-4 años para confirmarla.

In [ ]:
st_estable['mes'] = st_estable['fecha'].dt.month
meses_label = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

media_mes = st_estable.groupby('mes')['tasa_desnut'].mean()
std_mes   = st_estable.groupby('mes')['tasa_desnut'].std()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras por mes
colores_mes = ['#c0392b' if v >= media_mes.mean() else '#185fa5' for v in media_mes.values]
axes[0].bar(range(1, 13), media_mes.values, color=colores_mes, edgecolor='white')
axes[0].errorbar(range(1, 13), media_mes.values, yerr=std_mes.values,
                 fmt='none', color='black', capsize=4, linewidth=1.5)
axes[0].axhline(media_mes.mean(), color='gray', linestyle='--', linewidth=1,
                label=f'Media anual: {media_mes.mean():.1f}%')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(meses_label)
axes[0].set_title('Tasa de desnutrición promedio por mes\n(rojo = sobre la media | barras de error = std)', fontsize=12)
axes[0].set_ylabel('Tasa desnutrición (%)')
axes[0].set_ylim(70, 100)
axes[0].legend(fontsize=9)

# Comparativa 2024 vs 2025 lado a lado
pivot = st_estable.pivot_table(values='tasa_desnut', index='mes',
                                columns=st_estable['fecha'].dt.year)
x_pos = np.arange(12)
width = 0.35
axes[1].bar(x_pos - width/2, pivot[2024].values, width, label='2024',
            color='#185fa5', edgecolor='white', alpha=0.85)
axes[1].bar(x_pos + width/2, pivot[2025].values, width, label='2025',
            color='#e67e22', edgecolor='white', alpha=0.85)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(meses_label)
axes[1].set_title('Comparativa mensual 2024 vs 2025', fontsize=12)
axes[1].set_ylabel('Tasa desnutrición (%)')
axes[1].set_ylim(70, 100)
axes[1].legend(framealpha=0.5)
plt.tight_layout()
plt.show()

print('Tasa promedio por mes (2024-2025):')
resumen_mes = pd.DataFrame({
    'Mes': meses_label,
    'Media (%)': media_mes.values.round(1),
    'Std': std_mes.values.round(1),
    'Sobre media': ['↑' if v >= media_mes.mean() else '↓' for v in media_mes.values]
})
print(resumen_mes.to_string(index=False))
print()
mes_alto = meses_label[media_mes.values.argmax()]
mes_bajo = meses_label[media_mes.values.argmin()]
print(f'Mes con tasa más alta  : {mes_alto} ({media_mes.max():.1f}%)')
print(f'Mes con tasa más baja  : {mes_bajo} ({media_mes.min():.1f}%)')
print()
print('Patrón observado: tasas más altas en oct-nov-dic-ene (último trimestre y comienzo de año)')
print('Posible relación con ciclos escolares, estaciones de lluvia/sequía o períodos de cosecha.')
print('NOTA: Con solo 2 años de datos esta estacionalidad es indicativa, no concluyente.')

---
## 5. Modelos de proyección

Se implementan tres enfoques de menor a mayor complejidad:

| Modelo | Descripción | Requisito |
|---|---|---|
| **Promedio móvil** | Baseline — promedia los últimos N meses | Solo numpy |
| **Regresión lineal** | Proyecta la tendencia detectada | sklearn |
| **SARIMA** | Modelo estadístico para series con estacionalidad | statsmodels |

**Limitación importante:** Con 24 puntos estables (2024-2025), los modelos producen
intervalos de confianza amplios. Las proyecciones deben interpretarse como
**indicadores de dirección**, no como predicciones precisas.

### 5a. Promedio móvil (baseline)

In [ ]:
def proyectar_ma(serie, window, steps, nombre='MA'):
    """Proyecta usando promedio móvil simple."""
    hist = list(serie.values)
    preds = []
    for _ in range(steps):
        pred = np.mean(hist[-window:])
        preds.append(pred)
        hist.append(pred)
    return preds

STEPS = 6  # proyectar 6 meses hacia adelante (ene-jun 2026)

preds_ma3 = proyectar_ma(st_estable['tasa_desnut'], window=3, steps=STEPS, nombre='MA(3)')
preds_ma6 = proyectar_ma(st_estable['tasa_desnut'], window=6, steps=STEPS, nombre='MA(6)')

# Fechas futuras
ultima_fecha = st['fecha'].max()
fechas_futuras = pd.date_range(start=ultima_fecha + pd.DateOffset(months=1),
                               periods=STEPS, freq='MS')
meses_futuros = [f.strftime('%Y-%m') for f in fechas_futuras]

print('Proyección Promedio Móvil — próximos 6 meses:')
df_ma = pd.DataFrame({
    'Mes': meses_futuros,
    'MA(3) %': [round(p, 1) for p in preds_ma3],
    'MA(6) %': [round(p, 1) for p in preds_ma6],
})
print(df_ma.to_string(index=False))

### 5b. Regresión lineal con tendencia

In [ ]:
# Ajustar sobre serie estable
X_train_t = st_estable['t'].values.reshape(-1, 1)
y_train_t = st_estable['tasa_desnut'].values

lr_model = LinearRegression().fit(X_train_t, y_train_t)

# Proyectar: los próximos 6 puntos siguen el índice t
t_max = st_estable['t'].max()
t_futuro = np.arange(t_max + 1, t_max + STEPS + 1).reshape(-1, 1)
preds_lr = lr_model.predict(t_futuro)

# Calcular intervalo de confianza empírico (±1.96 * residual std)
residuos = y_train_t - lr_model.predict(X_train_t)
std_residuos = np.std(residuos)
ic_95 = 1.96 * std_residuos

print('Proyección Regresión Lineal — próximos 6 meses:')
df_lr = pd.DataFrame({
    'Mes'      : meses_futuros,
    'Pred (%)'  : [round(p, 1) for p in preds_lr],
    'IC inf (%)': [round(p - ic_95, 1) for p in preds_lr],
    'IC sup (%)': [round(min(p + ic_95, 100), 1) for p in preds_lr],
})
print(df_lr.to_string(index=False))
print(f'\nResidual std: {std_residuos:.2f}% | IC 95%: ±{ic_95:.1f} puntos porcentuales')
print(f'Pendiente   : {lr_model.coef_[0]:+.4f}%/mes')

### 5c. SARIMA (statsmodels)

In [ ]:
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.stattools import adfuller

    serie_sarima = st_estable['tasa_desnut'].values

    # Test de estacionariedad Dickey-Fuller
    adf_stat, adf_p, *_ = adfuller(serie_sarima)
    print(f'Test Dickey-Fuller: estadístico={adf_stat:.4f}, p-value={adf_p:.4f}')
    print(f'Serie {"estacionaria ✅" if adf_p < 0.05 else "NO estacionaria — se diferenciará (d=1)"}')

    # Ajustar SARIMA(1,1,1)(1,0,1,12)
    # p=1 (AR), d=1 (diferenciación), q=1 (MA), P=1, D=0, Q=1, s=12 (estacionalidad anual)
    modelo_sarima = SARIMAX(
        serie_sarima,
        order=(1, 1, 1),
        seasonal_order=(1, 0, 1, 12),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    resultado_sarima = modelo_sarima.fit(disp=False)
    print(f'\nSARIMA(1,1,1)(1,0,1,12) ajustado ✅')
    print(f'AIC: {resultado_sarima.aic:.2f} | BIC: {resultado_sarima.bic:.2f}')

    # Proyectar
    forecast_sarima = resultado_sarima.get_forecast(steps=STEPS)
    preds_sarima    = forecast_sarima.predicted_mean
    ic_sarima       = forecast_sarima.conf_int(alpha=0.05)

    print('\nProyección SARIMA — próximos 6 meses:')
    df_sarima = pd.DataFrame({
        'Mes'      : meses_futuros,
        'Pred (%)'  : preds_sarima.round(1).values,
        'IC inf (%)': ic_sarima.iloc[:, 0].round(1).values,
        'IC sup (%)': ic_sarima.iloc[:, 1].clip(upper=100).round(1).values,
    })
    print(df_sarima.to_string(index=False))
    SARIMA_DISPONIBLE = True

except ImportError:
    print('statsmodels no disponible en este entorno.')
    print('Instalar con: pip install statsmodels')
    print('Se usarán solo los modelos de Promedio Móvil y Regresión Lineal.')
    SARIMA_DISPONIBLE = False

---
## 6. Proyección 2026 — próximos 6 meses

Comparativa visual de todos los modelos disponibles con sus intervalos de confianza.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Historia reciente (2024-2025)
idx_2024 = st[st['anio_mes'] == '2024-01'].index[0]
historia_x = list(range(idx_2024, len(st)))
historia_y = st.loc[idx_2024:, 'tasa_desnut'].values
ax.plot(historia_x, historia_y, 'o-', color='#2c3e50', linewidth=2,
        markersize=5, label='Histórico 2024-2025', zorder=5)

# Punto de corte
ax.axvline(len(st) - 0.5, color='gray', linestyle=':', linewidth=1.5)
ax.text(len(st) - 0.3, 108, 'Proyección →', fontsize=9, color='gray')

# Eje x futuro
x_fut = list(range(len(st), len(st) + STEPS))

# MA(6)
ax.plot(x_fut, preds_ma6, 's--', color='#27ae60', linewidth=1.8,
        markersize=7, label='Promedio Móvil MA(6)', zorder=4)

# Regresión lineal + IC
ax.plot(x_fut, preds_lr, 'D--', color='#e67e22', linewidth=1.8,
        markersize=7, label=f'Regresión Lineal ({lr_model.coef_[0]:+.2f}%/mes)', zorder=4)
ax.fill_between(x_fut,
                [p - ic_95 for p in preds_lr],
                [min(p + ic_95, 100) for p in preds_lr],
                alpha=0.15, color='#e67e22', label=f'IC 95% reg. lineal (±{ic_95:.1f}%)')

# SARIMA si disponible
if SARIMA_DISPONIBLE:
    ax.plot(x_fut, preds_sarima.values, '^--', color='#8e44ad', linewidth=1.8,
            markersize=7, label='SARIMA(1,1,1)(1,0,1,12)', zorder=4)
    ax.fill_between(x_fut,
                    ic_sarima.iloc[:, 0].values,
                    ic_sarima.iloc[:, 1].clip(upper=100).values,
                    alpha=0.12, color='#8e44ad', label='IC 95% SARIMA')

# Umbrales de referencia
ax.axhline(90, color='#c0392b', linestyle='--', linewidth=0.8, alpha=0.6,
           label='Umbral crítico 90%')
ax.axhline(85, color='#e67e22', linestyle='--', linewidth=0.8, alpha=0.6,
           label='Promedio histórico ~85%')

# Etiquetas eje x
todos_x = historia_x + x_fut
todas_labels = (list(st.loc[idx_2024:, 'anio_mes']) + meses_futuros)
ax.set_xticks(todos_x)
ax.set_xticklabels(todas_labels, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Tasa de desnutrición (%)')
ax.set_ylim(65, 115)
ax.set_title('Proyección de la tasa de desnutrición — 2026\n'
             '(basada en datos 2024-2025)', fontsize=13, pad=12)
ax.legend(loc='lower left', framealpha=0.6, fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

print('\nResumen de proyecciones para los próximos 6 meses:')
df_resumen = pd.DataFrame({'Mes': meses_futuros, 'MA(6)': [round(p,1) for p in preds_ma6],
                            'Reg.Lineal': [round(p,1) for p in preds_lr]})
if SARIMA_DISPONIBLE:
    df_resumen['SARIMA'] = preds_sarima.round(1).values
print(df_resumen.to_string(index=False))
print(f'\nInterpretación: La tasa proyectada se mantiene en el rango {min(preds_ma6):.0f}%–{max(preds_ma6):.0f}%,')
print('consistente con el patrón histórico. No se proyecta mejora significativa.')

---
## 7. Análisis comparativo territorial

Se comparan los tres departamentos con datos suficientes:
**Cesar**, **La Guajira** y **Magdalena**.

Los demás departamentos (Santander, Zulia, Bogotá, etc.) tienen muy pocos casos
y sus tasas no son representativas estadísticamente.

In [ ]:
# Filtrar solo departamentos con datos suficientes
dptos_principales = ['CESAR', 'GUAJIRA', 'MAGDALENA']
sd_main = sd[sd['departamento'].isin(dptos_principales)].copy()

COLORES_DPTO = {'CESAR': '#185fa5', 'GUAJIRA': '#c0392b', 'MAGDALENA': '#27ae60'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── 1. Tasa de desnutrición por dpto y año ────────────────────────────────
anios = sorted(sd_main['anio'].unique())
x = np.arange(len(dptos_principales))
width = 0.25
ax = axes[0, 0]
for i, anio in enumerate(anios):
    vals = []
    for dpto in dptos_principales:
        fila = sd_main[(sd_main['anio'] == anio) & (sd_main['departamento'] == dpto)]
        vals.append(fila['tasa_desnut'].values[0] if len(fila) > 0 else np.nan)
    offset = (i - (len(anios)-1)/2) * width
    ax.bar(x + offset, vals, width, label=str(anio), edgecolor='white', alpha=0.85)
    for j, v in enumerate(vals):
        if not np.isnan(v):
            ax.text(x[j] + offset, v + 0.5, f'{v:.0f}%', ha='center', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(dptos_principales)
ax.set_title('Tasa de desnutrición por departamento y año', fontsize=11)
ax.set_ylabel('Tasa (%)')
ax.set_ylim(0, 105)
ax.legend(title='Año', fontsize=9)

# ── 2. Tasa de desnutrición severa ────────────────────────────────────────
ax = axes[0, 1]
for i, anio in enumerate(anios):
    vals = []
    for dpto in dptos_principales:
        fila = sd_main[(sd_main['anio'] == anio) & (sd_main['departamento'] == dpto)]
        vals.append(fila['tasa_severa'].values[0] if len(fila) > 0 else np.nan)
    offset = (i - (len(anios)-1)/2) * width
    ax.bar(x + offset, vals, width, label=str(anio), edgecolor='white', alpha=0.85)
    for j, v in enumerate(vals):
        if not np.isnan(v):
            ax.text(x[j] + offset, v + 0.2, f'{v:.0f}%', ha='center', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(dptos_principales)
ax.set_title('Tasa de desnutrición SEVERA por departamento y año', fontsize=11)
ax.set_ylabel('Tasa severa (%)')
ax.legend(title='Año', fontsize=9)

# ── 3. Z-score promedio ───────────────────────────────────────────────────
ax = axes[1, 0]
for dpto in dptos_principales:
    sub = sd_main[sd_main['departamento'] == dpto].sort_values('anio')
    ax.plot(sub['anio'], sub['zscore_medio'], 'o-',
            color=COLORES_DPTO[dpto], linewidth=2, markersize=8, label=dpto)
    for _, row in sub.iterrows():
        ax.annotate(f'{row["zscore_medio"]:.2f}',
                    (row['anio'], row['zscore_medio']),
                    textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)
ax.axhline(-2, color='#e67e22', linestyle='--', linewidth=1, label='Umbral desnut. moderada')
ax.axhline(-3, color='#c0392b', linestyle='--', linewidth=1, label='Umbral severa')
ax.set_title('Evolución del z-score promedio por departamento', fontsize=11)
ax.set_ylabel('Z-score promedio')
ax.set_xticks(anios)
ax.legend(fontsize=8, ncol=2)

# ── 4. Peso promedio actual ───────────────────────────────────────────────
ax = axes[1, 1]
for dpto in dptos_principales:
    sub = sd_main[sd_main['departamento'] == dpto].sort_values('anio')
    ax.plot(sub['anio'], sub['peso_act_med'], 'o-',
            color=COLORES_DPTO[dpto], linewidth=2, markersize=8, label=dpto)
    for _, row in sub.iterrows():
        ax.annotate(f'{row["peso_act_med"]:.1f}',
                    (row['anio'], row['peso_act_med']),
                    textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)
ax.set_title('Peso promedio actual por departamento (kg)', fontsize=11)
ax.set_ylabel('Peso promedio (kg)')
ax.set_xticks(anios)
ax.legend(fontsize=9)

plt.suptitle('Análisis comparativo territorial — Cesar, La Guajira y Magdalena',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla resumen comparativa
print('=== TABLA COMPARATIVA TERRITORIAL ===')
print()
for dpto in dptos_principales:
    sub = sd_main[sd_main['departamento'] == dpto].sort_values('anio')
    print(f'── {dpto} ──')
    print(sub[['anio','casos','tasa_desnut','tasa_severa','zscore_medio','peso_act_med']].to_string(index=False))
    print()

print('=== HALLAZGOS TERRITORIALES ===')
print()
# Calcular diferencias clave
for anio in [2024, 2025]:
    cesar   = sd_main[(sd_main['departamento']=='CESAR')   & (sd_main['anio']==anio)]
    guajira = sd_main[(sd_main['departamento']=='GUAJIRA') & (sd_main['anio']==anio)]
    if len(cesar) > 0 and len(guajira) > 0:
        diff_tasa = guajira['tasa_desnut'].values[0] - cesar['tasa_desnut'].values[0]
        diff_z    = guajira['zscore_medio'].values[0] - cesar['zscore_medio'].values[0]
        print(f'{anio}: La Guajira supera al Cesar en {diff_tasa:+.1f} puntos de tasa de desnutrición')
        print(f'       Z-score La Guajira: {guajira["zscore_medio"].values[0]:.3f} vs Cesar: {cesar["zscore_medio"].values[0]:.3f}')
        print()

---
## 8. Hallazgos y limitaciones

### Hallazgos principales

| Hallazgo | Evidencia |
|---|---|
| Tasa estable ~85-87% en 2024-2025 | Sin mejora ni deterioro significativo |
| Tendencia levemente creciente | Pendiente ~+0.22%/mes, pero p>0.05 (no significativo) |
| Pico estacional en oct-nov-dic-ene | Patrón consistente en 2024 y 2025 |
| La Guajira supera al Cesar | ~91-94% vs ~75-79% de tasa de desnutrición |
| Magdalena muestra altas tasas de severidad | ~41-44% de casos severos en 2024-2025 |
| Z-score promedio ~-2.3 | Desnutrición moderada sostenida como tendencia central |

### Proyección 2026

La tasa de desnutrición para los primeros 6 meses de 2026 se proyecta en el rango
**87%–90%**, consistente con el patrón histórico. No se proyecta mejora sin
intervenciones estructurales.

### Limitaciones metodológicas

1. **Solo 36 puntos mensuales** (24 estables) — insuficiente para modelos complejos
2. **2023 inestable** — captura parcial de datos ese año distorsiona el inicio de la serie
3. **Intervalos de confianza amplios** — las proyecciones tienen incertidumbre de ±8-10 puntos
4. **Sin variables exógenas** — no se incorporaron factores como lluvias, cosechas o programas sociales
5. **Datos de una sola IPS** — la serie refleja la demanda de atención, no la prevalencia poblacional real

### Próximos pasos
- **Notebook 07:** Dashboard interactivo de visualización de resultados
- Con más años de datos, SARIMA o Prophet darían proyecciones más confiables
- Incorporar datos del SIVIGILA departamental completo ampliaría la serie territorial